# Inferensi dan Verifikasi Hasil Babak Penyisihan

Big Data Challenge Satria Data 2026

Notebook ini hanya menjalankan inferensi, tanpa pelatihan. *Checkpoint* diambil dari Hugging Face, lalu prediksinya dibandingkan dengan berkas CSV yang dikumpulkan saat *submit*. Tujuannya menunjukkan bahwa hasil yang dikumpulkan dapat dibangkitkan kembali dari model yang dilaporkan hanya dengan citra data uji, tanpa label data uji.

Langkah kerja:

1. Menyiapkan lingkungan Kaggle dan memuat pustaka.
2. Membaca citra data uji dari dataset Kaggle.
3. Mengunduh *checkpoint* dari Hugging Face, lalu memeriksa ukuran, SHA-256, dan metadata di dalamnya.
4. Membangun arsitektur yang sama dengan notebook pelatihan dan memuat bobot.
5. Menjalankan inferensi dengan *test-time augmentation* (TTA) tiga *view*.
6. Memuat CSV hasil *submit*.
7. Membandingkan prediksi hasil reproduksi dengan CSV *submit*.
8. Menyimpan laporan verifikasi.

Pengaturan Kaggle yang diperlukan (panel *Notebook options* di sisi kanan):

- *Accelerator*: GPU T4 x2, sama seperti saat pelatihan. Notebook ini hanya memakai satu GPU.
- *Input* model: tambahkan model SigLIP 2 (`siglip2-400m-224`) yang sama dengan notebook pelatihan. Jika tidak ditemukan, model dasar diunduh dari Hugging Face.
- *Internet*: On, untuk mengunduh *checkpoint* `train_val_best_test.pt` dari Hugging Face dan CSV *submit* dari Google Drive.
- *Input*: tambahkan dataset `bdc-2026` yang berisi folder `test/`, sama seperti notebook pelatihan.
- Repositori Hugging Face `ayaayaa/train-val-best-test` bersifat publik. Jika privat, simpan token pada *Add-ons*, *Secrets* dengan nama `HF_TOKEN`.
- CSV hasil *submit* diambil dari Google Drive. Jika tidak dapat diunduh, tambahkan CSV tersebut sebagai *Input* dataset Kaggle dan isi `SUBMITTED_CSV_LOCAL` pada sel konfigurasi.

## 1. Persiapan Lingkungan

Sel berikut memasang `gdown` untuk mengunduh CSV dari Google Drive. Lingkungan Kaggle yang dipakai pada pelatihan memakai `transformers` 5.0.0 dan PyTorch 2.10, sehingga notebook ini tidak menurunkan atau mengganti versi pustaka tersebut. Jika pemasangan `gdown` gagal, notebook tetap berjalan dan CSV *submit* dapat dimuat dari dataset Kaggle.

In [1]:
import subprocess
import sys

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    print("gdown terpasang.")
except Exception as exc:
    print("Pemasangan gdown gagal (Internet Kaggle mungkin Off):", exc)

gdown terpasang.


## 2. Impor Pustaka dan Konfigurasi

Parameter inferensi disamakan dengan notebook pelatihan:

- Resolusi 224 × 224 piksel, dengan rata-rata dan simpangan baku normalisasi 0,5 (nilai dari *processor* SigLIP 2).
- Warna *padding* (124, 116, 104), sama dengan yang dipakai pada pelatihan.
- TTA tiga *view*: citra asli, *flip* horizontal, dan *center crop* 90%.
- *Classifier head* hierarkis dengan 512 unit tersembunyi dan *dropout* 0,3.
- Presisi campuran FP16 pada *encoder*, sedangkan *head* berjalan dalam FP32.

Jalur data uji mengikuti notebook pelatihan. Ubah `TEST_DIR` jika nama dataset berbeda.

In [2]:
import os
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")   # cache unduhan model

import gc
import hashlib
import json
import os
import platform
import random
import shutil
import zipfile
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from IPython.display import display
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# ---- Sumber model ----
HF_REPO_ID = "ayaayaa/train-val-best-test"
HF_REPO_TYPE = "dataset"
HF_CKPT_FILENAME = "train_val_best_test.pt"
BASE_MODEL_ID = "google/siglip2-so400m-patch14-224"   # cadangan bila model lokal tidak ada

# ---- Model dasar SigLIP 2 pada Input Kaggle (kandidat yang sama dengan notebook pelatihan) ----
_MODEL_CANDIDATES = [
    Path("/kaggle/input/models/user235162736/siglip2-400m-224/pytorch/default/1"),
    Path("/kaggle/input/siglip2-so400m-patch14-224"),
    Path("/kaggle/input/siglip2-400-224"),
    Path("/kaggle/input/siglip2_400_224"),
    Path("/kaggle/input/datasets/user235162736/bdc-2026/siglip2_400_224"),
]
BASE_MODEL_LOCAL_DIR = next(
    (p for p in _MODEL_CANDIDATES if (p / "config.json").exists() and (p / "model.safetensors").exists()),
    None,
)
BASE_MODEL_SOURCE = str(BASE_MODEL_LOCAL_DIR) if BASE_MODEL_LOCAL_DIR else BASE_MODEL_ID

# ---- CSV hasil submit (Google Drive) ----
SUBMITTED_CSV_DRIVE_ID = "1016fBakMpfRiHCsTYohpcvnMMXME5jDg"
SUBMITTED_CSV_PATH = "/kaggle/working/submission_asli.csv"
# Jika CSV submit ditambahkan sebagai Input dataset Kaggle, isi jalurnya di sini, mis.
# "/kaggle/input/<nama-dataset>/submission.csv". Jika None, CSV diunduh dari Google Drive.
SUBMITTED_CSV_LOCAL = None

# ---- Data uji (dataset Kaggle yang sama dengan notebook pelatihan) ----
TEST_DIR = Path("/kaggle/input/datasets/user235162736/bdc-2026/BDC2026/BDC2026/test")
EXPECTED_TEST_IMAGES = 1458
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}

# ---- Parameter inferensi (sama dengan notebook pelatihan) ----
IMAGE_SIZE = 224
VISION_MEAN = [0.5, 0.5, 0.5]
VISION_STD = [0.5, 0.5, 0.5]
PADDING_VALUE = (124, 116, 104)
TTA_N_VIEWS = 3
HIDDEN_SIZE = 1152
HEAD_HIDDEN_DIM = 512
HEAD_DROPOUT = 0.3
CLASS_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}
SEED = 1000

# ---- Pengaturan eksekusi (tidak memengaruhi hasil selain selisih numerik yang sangat kecil) ----
BATCH_SIZE = 64
NUM_WORKERS = 4

OUTPUT_DIR = Path("/kaggle/working/hasil_verifikasi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def fmt_pct(value: float, decimals: int = 4) -> str:
    """Format 0-1 sebagai persen dengan koma desimal, mis. 0.9966 -> '99,6600'."""
    return f"{value * 100:.{decimals}f}".replace(".", ",")


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("Perangkat   :", DEVICE, "|", GPU_NAME)
print("Model dasar :", BASE_MODEL_SOURCE, "(folder Input Kaggle)" if BASE_MODEL_LOCAL_DIR else "(unduh dari Hugging Face)")
if DEVICE.type != "cuda":
    print("PERINGATAN: GPU tidak aktif. Aktifkan Accelerator GPU T4 x2 pada Notebook options.")

Python      : 3.12.13
PyTorch     : 2.10.0+cu128
Transformers: 5.0.0
Perangkat   : cuda | Tesla T4
Model dasar : /kaggle/input/models/user235162736/siglip2-400m-224/pytorch/default/1 (folder Input Kaggle)


## 3. Persiapan Data Uji

Data uji dibaca dari dataset Kaggle `bdc-2026`. Langkah pada bagian ini:

1. Memeriksa folder `TEST_DIR`. Jika tidak ditemukan, notebook mencari folder `test` lain di `/kaggle/input` yang berisi citra bernama angka.
2. Mengumpulkan citra yang nama berkasnya berupa angka. Angka tersebut menjadi `id`.
3. Memeriksa jumlah citra dan memastikan tidak ada `id` ganda.

Tidak ada label yang dibaca pada tahap ini. Notebook hanya memuat citra dan `id`.

In [4]:
def has_numeric_images(folder: Path, limit: int = 50) -> bool:
    count = 0
    for f in folder.iterdir():
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS and f.stem.isdigit():
            return True
        count += 1
        if count >= limit:
            break
    return False


def find_test_dir(preferred: Path) -> Optional[Path]:
    if preferred.is_dir():
        return preferred
    print(f"Folder {preferred} tidak ditemukan. Mencari folder test lain di /kaggle/input ...")
    for dirpath, dirnames, _ in os.walk("/kaggle/input"):
        if Path(dirpath).name.lower() == "test" and has_numeric_images(Path(dirpath)):
            return Path(dirpath)
    return None


test_root = find_test_dir(TEST_DIR)
if test_root is None:
    raise FileNotFoundError(
        "Folder data uji tidak ditemukan. Tambahkan dataset bdc-2026 melalui Add Input, "
        "lalu sesuaikan TEST_DIR pada sel konfigurasi."
    )

id_to_path: dict[int, Path] = {}
duplicate_ids: list[int] = []
for p in sorted(test_root.rglob("*")):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and p.stem.isdigit():
        image_id = int(p.stem)
        if image_id in id_to_path:
            duplicate_ids.append(image_id)
        id_to_path[image_id] = p

if not id_to_path:
    raise FileNotFoundError(f"Tidak ada citra dengan nama berkas angka di {test_root}")
if duplicate_ids:
    raise RuntimeError(f"ID ganda ditemukan pada data uji, contoh: {duplicate_ids[:10]}")

sorted_ids = sorted(id_to_path)
test_df = pd.DataFrame({
    "id": sorted_ids,
    "image_path": [str(id_to_path[i]) for i in sorted_ids],
})
print("Folder data uji  :", test_root)
print("Jumlah citra uji :", len(test_df))
if len(test_df) != EXPECTED_TEST_IMAGES:
    print(f"Catatan: jumlah citra uji pada babak penyisihan adalah {EXPECTED_TEST_IMAGES}.")
display(test_df.head())

Folder data uji  : /kaggle/input/datasets/user235162736/bdc-2026/BDC2026/BDC2026/test
Jumlah citra uji : 1458


,id,image_path
0,1,/kaggle/input/datasets/user235162736/bdc-2026/...
1,2,/kaggle/input/datasets/user235162736/bdc-2026/...
2,3,/kaggle/input/datasets/user235162736/bdc-2026/...
3,4,/kaggle/input/datasets/user235162736/bdc-2026/...
4,5,/kaggle/input/datasets/user235162736/bdc-2026/...


## 4. Unduh *Checkpoint* dari Hugging Face

Langkahnya:

1. Mengunduh `train_val_best_test.pt` dari repositori dataset `ayaayaa/train-val-best-test`.
2. Menghitung SHA-256 berkas sebagai sidik jari *checkpoint*, sehingga pihak lain dapat memastikan berkas yang sama dipakai.
3. Memuat berkas dan menampilkan metadata yang tersimpan di dalamnya: *epoch*, *macro-F1* validasi, dan konfigurasi pelatihan.

*Checkpoint* ini berisi tiga kelompok bobot: `encoder_state_dict`, `attn_pool_state_dict`, dan `head_state_dict`.

In [5]:
from huggingface_hub import hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None   # repositori publik tidak memerlukan token

ckpt_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=HF_CKPT_FILENAME,
    repo_type=HF_REPO_TYPE,
    token=HF_TOKEN,
)


def sha256_file(path: str, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


ckpt_size_bytes = os.path.getsize(ckpt_path)
ckpt_sha256 = sha256_file(ckpt_path)
print("Berkas    :", ckpt_path)
print(f"Ukuran    : {ckpt_size_bytes / 1024**3:.3f} GB")
print("SHA-256   :", ckpt_sha256)

train_val_best_test.pt:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Berkas    : /kaggle/working/hf_cache/hub/datasets--ayaayaa--train-val-best-test/snapshots/1bc33974729ec03086ebde5e528a22128333acc5/train_val_best_test.pt
Ukuran    : 1.617 GB
SHA-256   : 03aa480d1568ee5b5303428b146c13970e810f308bb6e39c5956949492acfe14


In [6]:
try:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
except Exception as exc:
    print("Pemuatan dengan weights_only=True gagal:", exc)
    print("Mencoba weights_only=False (berkas ini milik sendiri).")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("Isi checkpoint:", list(ckpt.keys()))
print("Epoch tersimpan          :", ckpt.get("epoch"))
print("Macro-F1 validasi        :", fmt_pct(ckpt["val_macro_f1"], 6) if ckpt.get("val_macro_f1") is not None else "N/A")
print("Akurasi validasi         :", fmt_pct(ckpt["val_accuracy"], 6) if ckpt.get("val_accuracy") is not None else "N/A")
print("Macro-F1 latih           :", fmt_pct(ckpt["train_macro_f1"], 6) if ckpt.get("train_macro_f1") is not None else "N/A")
print("Dimensi fitur            :", ckpt.get("feature_dim"))
assert ckpt.get("feature_dim") == 2 * HIDDEN_SIZE, "Dimensi fitur tidak sesuai dengan arsitektur."

train_config = ckpt.get("config", {})
print("\nKonfigurasi pelatihan yang tersimpan di checkpoint:")
print(json.dumps(train_config, indent=2, ensure_ascii=False, default=str))

Isi checkpoint: ['encoder_state_dict', 'head_state_dict', 'attn_pool_state_dict', 'feature_dim', 'epoch', 'train_loss', 'train_accuracy', 'train_macro_f1', 'val_accuracy', 'val_macro_f1', 'test_macro_f1', 'test_accuracy', 'config']
Epoch tersimpan          : 4
Macro-F1 validasi        : 98,872756
Akurasi validasi         : 98,718432
Macro-F1 latih           : 99,634661
Dimensi fitur            : 2304

Konfigurasi pelatihan yang tersimpan di checkpoint:
{
  "model_name": "/kaggle/input/models/user235162736/siglip2-400m-224/pytorch/default/1",
  "hf_model_repo_id": "google/siglip2-so400m-patch14-224",
  "local_model_dir": "/kaggle/input/models/user235162736/siglip2-400m-224/pytorch/default/1",
  "image_size": 224,
  "preprocessing_version": "exif_rgb_fit_pad_224_siglip2_so400m_norm_v1",
  "augmentation_version": "pil_crop_flip_rotate_translate_color_blur_gray_cutout_v2",
  "augs_per_image": 4,
  "include_original_in_head_training": true,
  "training_strategy": "train_val_split",
  "val_s

## 5. Arsitektur Model dan Pemuatan Bobot

Definisi kelas pada bagian ini sama dengan notebook pelatihan, termasuk nama parameter, sehingga bobot dapat dimuat dengan `strict=True`. Langkahnya:

1. Memuat SigLIP 2 SO400M dari folder Input Kaggle (atau dari Hugging Face bila folder tidak ditemukan), lalu membuang menara teks dan proyeksi multimodal. Bobot awal ini segera ditimpa oleh `encoder_state_dict` dari *checkpoint*.
2. Membangun `AttentionPool` dan `HierarchicalClassifierHead`, lalu memuat bobotnya dari *checkpoint*.
3. Menyusun `SiglipFeatureExtractor` yang menggabungkan *pooled output* (1.152) dengan hasil *attention pooling* (1.152) menjadi fitur 2.304 dimensi.
4. Memindahkan model ke GPU dan mengaturnya ke mode evaluasi.

Jika pemuatan bobot gagal karena perbedaan nama parameter, periksa versi `transformers` yang tercetak pada bagian 2. Pelatihan memakai versi 5.0.0.

In [7]:
class AttentionPool(nn.Module):
    """Attention pooling dengan satu query yang dapat dipelajari atas token patch."""

    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = int(hidden_size)
        self.query = nn.Parameter(torch.zeros(1, 1, self.hidden_size))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.out_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.scale = self.hidden_size ** -0.5

    def forward(self, patch_tokens: torch.Tensor) -> torch.Tensor:
        batch = patch_tokens.shape[0]
        query = self.query.expand(batch, -1, -1).to(patch_tokens.dtype)   # (B, 1, D)
        keys = self.k_proj(patch_tokens)                                  # (B, N, D)
        values = self.v_proj(patch_tokens)                                # (B, N, D)
        attn_logits = torch.matmul(query, keys.transpose(-2, -1)) * self.scale
        attn_weights = attn_logits.softmax(dim=-1)
        pooled = torch.matmul(attn_weights, values)                       # (B, 1, D)
        return self.out_proj(pooled.squeeze(1))                           # (B, D)


class HierarchicalClassifierHead(nn.Module):
    """Dua head biner: tahap 1 (Electronic vs bukan) dan tahap 2 (Recyclable vs Organic)."""

    def __init__(self, feature_dim: int, hidden_dim: int = HEAD_HIDDEN_DIM, dropout: float = HEAD_DROPOUT) -> None:
        super().__init__()

        def _make_binary_head() -> nn.Sequential:
            return nn.Sequential(
                nn.LayerNorm(feature_dim),
                nn.Linear(feature_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 2),
            )

        self.electronic_head = _make_binary_head()
        self.recyclable_organic_head = _make_binary_head()

    def forward(self, features: torch.Tensor):
        return self.electronic_head(features), self.recyclable_organic_head(features)


def hierarchical_probs(electronic_logits: torch.Tensor, ro_logits: torch.Tensor) -> torch.Tensor:
    """Gabungkan dua head menjadi probabilitas [Recyclable, Electronic, Organic]."""
    p_electronic = F.softmax(electronic_logits, dim=1)
    p_ro = F.softmax(ro_logits, dim=1)
    p_not_electronic = p_electronic[:, 0]
    p_e = p_electronic[:, 1]
    p_recyclable = p_not_electronic * p_ro[:, 0]
    p_organic = p_not_electronic * p_ro[:, 1]
    return torch.stack([p_recyclable, p_e, p_organic], dim=1)


class SiglipFeatureExtractor(nn.Module):
    def __init__(self, encoder_model: nn.Module, hidden_size: int, attn_pool: nn.Module) -> None:
        super().__init__()
        self.encoder_model = encoder_model
        self.hidden_size = int(hidden_size)
        self.attn_pool = attn_pool

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        use_amp = pixel_values.is_cuda
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            outputs = self.encoder_model.vision_model(
                pixel_values=pixel_values, output_hidden_states=False, return_dict=True
            )
            tokens = outputs.last_hidden_state
            pooled = outputs.pooler_output if outputs.pooler_output is not None else tokens.mean(dim=1)
            attn_patch = self.attn_pool(tokens)
            return torch.cat([pooled, attn_patch], dim=1)

In [8]:
from transformers import AutoModel

print(f"Memuat model dasar dari: {BASE_MODEL_SOURCE}")
try:
    encoder = AutoModel.from_pretrained(BASE_MODEL_SOURCE, dtype=torch.float32)
except TypeError:
    encoder = AutoModel.from_pretrained(BASE_MODEL_SOURCE, torch_dtype=torch.float32)
encoder = encoder.float()

# Menara teks dan proyeksi multimodal dibuang, sama seperti pada notebook pelatihan.
for attr in ["text_model", "text_projection", "vision_projection", "logit_scale", "logit_bias"]:
    if hasattr(encoder, attr):
        delattr(encoder, attr)
gc.collect()

encoder.load_state_dict(ckpt["encoder_state_dict"])      # strict=True
attn_pool = AttentionPool(HIDDEN_SIZE)
attn_pool.load_state_dict(ckpt["attn_pool_state_dict"])
head = HierarchicalClassifierHead(2 * HIDDEN_SIZE)
head.load_state_dict(ckpt["head_state_dict"])

feature_extractor = SiglipFeatureExtractor(encoder, HIDDEN_SIZE, attn_pool).to(DEVICE).eval()
head = head.to(DEVICE).eval()

del ckpt
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

n_encoder = sum(p.numel() for p in encoder.parameters())
n_pool = sum(p.numel() for p in attn_pool.parameters())
n_head = sum(p.numel() for p in head.parameters())
print(f"Parameter encoder        : {n_encoder:,}")
print(f"Parameter attention pool : {n_pool:,}")
print(f"Parameter classifier head: {n_head:,}")
print("Bobot dari checkpoint berhasil dimuat (strict=True).")

# Pemeriksaan nilai normalisasi terhadap processor SigLIP 2.
try:
    from transformers import AutoImageProcessor
    processor = AutoImageProcessor.from_pretrained(BASE_MODEL_SOURCE)
    same = list(processor.image_mean) == VISION_MEAN and list(processor.image_std) == VISION_STD
    print("Mean/std processor       :", list(processor.image_mean), list(processor.image_std),
          "| sesuai konfigurasi" if same else "| BERBEDA dari konfigurasi")
except Exception as exc:
    print("Pemeriksaan processor dilewati:", exc)

Memuat model dasar dari: /kaggle/input/models/user235162736/siglip2-400m-224/pytorch/default/1


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Parameter encoder        : 427,680,704
Parameter attention pool : 3,985,920
Parameter classifier head: 2,371,588
Bobot dari checkpoint berhasil dimuat (strict=True).
Mean/std processor       : [0.5, 0.5, 0.5] [0.5, 0.5, 0.5] | sesuai konfigurasi


## 6. Praproses Citra dan *View* TTA

Praproses sama dengan notebook pelatihan:

1. Koreksi orientasi EXIF dan konversi ke RGB.
2. Pembentukan *view*: 0 = citra asli, 1 = *flip* horizontal, 2 = *center crop* 90%.
3. Penyesuaian ukuran dengan mempertahankan rasio aspek (interpolasi *bicubic*), lalu *padding* menjadi 224 × 224 piksel.
4. Normalisasi dengan rata-rata dan simpangan baku 0,5.

Semua langkah bersifat deterministik, sehingga citra yang sama selalu menghasilkan tensor yang sama.

In [9]:
def resize_and_pad(image: Image.Image, size: int, fill: tuple) -> Image.Image:
    image = image.convert("RGB")
    width, height = image.size
    scale = min(size / width, size / height)
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))
    resized = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
    canvas = Image.new("RGB", (size, size), fill)
    canvas.paste(resized, ((size - new_w) // 2, (size - new_h) // 2))
    return canvas


def preprocess_tta_view(image: Image.Image, view_idx: int) -> torch.Tensor:
    image = ImageOps.exif_transpose(image).convert("RGB")
    if view_idx == 0:
        pass
    elif view_idx == 1:
        image = ImageOps.mirror(image)
    elif view_idx == 2:
        width, height = image.size
        crop_w, crop_h = int(round(width * 0.9)), int(round(height * 0.9))
        left = (width - crop_w) // 2
        top = (height - crop_h) // 2
        image = image.crop((left, top, left + crop_w, top + crop_h))
    else:
        raise ValueError(f"view_idx={view_idx} tidak didukung (hanya 0, 1, 2)")
    image = resize_and_pad(image, size=IMAGE_SIZE, fill=PADDING_VALUE)
    tensor = transforms.functional.to_tensor(image)
    return transforms.functional.normalize(tensor, mean=VISION_MEAN, std=VISION_STD)


class TestImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, view_idx: int) -> None:
        self.frame = frame.reset_index(drop=True)
        self.view_idx = view_idx

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        path = Path(str(row.image_path))
        try:
            with Image.open(path) as image:
                tensor = preprocess_tta_view(image, self.view_idx)
        except Exception as exc:
            raise RuntimeError(f"Gagal memproses citra {path}: {exc}") from exc
        return tensor, int(row.id)


# Pemeriksaan bentuk tensor pada satu citra.
with Image.open(test_df.iloc[0].image_path) as im:
    for v in range(TTA_N_VIEWS):
        assert preprocess_tta_view(im, v).shape == (3, IMAGE_SIZE, IMAGE_SIZE)
print("Pemeriksaan praproses berhasil untuk", TTA_N_VIEWS, "view.")

Pemeriksaan praproses berhasil untuk 3 view.


## 7. Inferensi dengan TTA

Langkahnya:

1. Untuk setiap *view* (0, 1, 2), seluruh citra uji diproses tanpa pengacakan urutan.
2. Setiap citra melalui *feature extractor* dan *classifier head*, lalu probabilitas tiga kelas dihitung dengan `hierarchical_probs`.
3. Probabilitas dirata-ratakan lintas *view*. Kelas dengan probabilitas tertinggi menjadi prediksi.
4. Prediksi *view* 0 saja (tanpa TTA) disimpan sebagai pembanding.
5. Hasil ditulis ke `submission_reproduksi.csv` (kolom `id`, `predicted`) dan `probabilitas_reproduksi.csv`.

Kolom `margin` adalah selisih probabilitas tertinggi dan kedua tertinggi. Nilai kecil berarti citra berada dekat batas keputusan.

In [10]:
@torch.inference_mode()
def predict_view(view_idx: int):
    dataset = TestImageDataset(test_df, view_idx)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    )
    probs_parts, ids = [], []
    for pixel_values, batch_ids in tqdm(loader, desc=f"view {view_idx}", leave=False):
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)
        combined = feature_extractor(pixel_values)
        electronic_logits, ro_logits = head(combined.float())
        probs = hierarchical_probs(electronic_logits.float(), ro_logits.float())
        probs_parts.append(probs.cpu())
        ids.extend(batch_ids.tolist())
    return torch.cat(probs_parts).numpy(), ids


view_probs, ids_ref = [], None
for view_idx in range(TTA_N_VIEWS):
    probs, ids = predict_view(view_idx)
    if ids_ref is None:
        ids_ref = ids
    assert ids == ids_ref, "Urutan id berbeda antar view."
    view_probs.append(probs)

per_view = np.stack(view_probs, axis=0)          # (n_views, N, 3)
avg_probs = per_view.mean(axis=0)
pred_tta = avg_probs.argmax(axis=1).astype(int)
pred_no_tta = per_view[0].argmax(axis=1).astype(int)
sorted_probs = np.sort(avg_probs, axis=1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]

result_df = pd.DataFrame({
    "id": ids_ref,
    "prob_recyclable": avg_probs[:, 0],
    "prob_electronic": avg_probs[:, 1],
    "prob_organic": avg_probs[:, 2],
    "pred_no_tta": pred_no_tta,
    "predicted": pred_tta,
    "margin": margin,
})

assert set(result_df["predicted"].unique()).issubset({0, 1, 2})
REPRO_CSV_PATH = OUTPUT_DIR / "submission_reproduksi.csv"
PROB_CSV_PATH = OUTPUT_DIR / "probabilitas_reproduksi.csv"
result_df[["id", "predicted"]].to_csv(REPRO_CSV_PATH, index=False)
result_df.to_csv(PROB_CSV_PATH, index=False)

n_changed_by_tta = int((pred_tta != pred_no_tta).sum())
print(f"Jumlah citra diprediksi   : {len(result_df)}")
print(f"Prediksi berubah oleh TTA : {n_changed_by_tta} dari {len(result_df)}")
print("Disimpan:", REPRO_CSV_PATH)
display(result_df[["id", "predicted"]].head())

view 0:   0%|          | 0/23 [00:00<?, ?it/s]

view 1:   0%|          | 0/23 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77fce0596d40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
AssertionError: can only test a child process


view 2:   0%|          | 0/23 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77fce0596d40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x77fce0596d40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Jumlah citra diprediksi   : 1458
Prediksi berubah oleh TTA : 2 dari 1458
Disimpan: /kaggle/working/hasil_verifikasi/submission_reproduksi.csv


,id,predicted
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0


## 8. Memuat CSV Hasil *Submit*

CSV yang dikumpulkan saat *submit* dimuat dengan urutan prioritas berikut:

1. Berkas lokal pada `SUBMITTED_CSV_LOCAL`, jika CSV ditambahkan sebagai *Input* dataset Kaggle.
2. Unduhan dari Google Drive dengan `gdown`, yang membutuhkan Internet Kaggle On dan berkas yang dibagikan untuk umum.

Jika kedua cara gagal, notebook berhenti dengan pesan yang menjelaskan langkah perbaikannya.

In [11]:
if SUBMITTED_CSV_LOCAL and Path(SUBMITTED_CSV_LOCAL).exists():
    shutil.copy(SUBMITTED_CSV_LOCAL, SUBMITTED_CSV_PATH)
    print("CSV submit dimuat dari:", SUBMITTED_CSV_LOCAL)
else:
    downloaded = None
    try:
        import gdown
        drive_url = f"https://drive.google.com/uc?id={SUBMITTED_CSV_DRIVE_ID}"
        downloaded = gdown.download(drive_url, SUBMITTED_CSV_PATH, quiet=False)
    except Exception as exc:
        print("Pengunduhan dari Google Drive gagal:", exc)

    if not downloaded or not os.path.exists(SUBMITTED_CSV_PATH) or os.path.getsize(SUBMITTED_CSV_PATH) == 0:
        raise FileNotFoundError(
            "CSV submit tidak dapat diunduh. Pastikan Internet Kaggle On dan berkas Drive dibagikan "
            "untuk umum, atau tambahkan CSV sebagai Input dataset Kaggle lalu isi SUBMITTED_CSV_LOCAL."
        )

submitted_raw = pd.read_csv(SUBMITTED_CSV_PATH)
print("Kolom CSV submit :", list(submitted_raw.columns))
print("Jumlah baris     :", len(submitted_raw))
display(submitted_raw.head())

Downloading...
From: https://drive.google.com/uc?id=1016fBakMpfRiHCsTYohpcvnMMXME5jDg
To: /kaggle/working/submission_asli.csv
100%|██████████| 9.11k/9.11k [00:00<00:00, 13.8MB/s]

Kolom CSV submit : ['id', 'predicted']
Jumlah baris     : 1458


,id,predicted
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0


## 9. Perbandingan dengan CSV *Submit*

Langkahnya:

1. Menyamakan format kolom menjadi `id` dan `predicted`, lalu memeriksa apakah kumpulan `id` kedua berkas sama.
2. Menghitung jumlah dan persentase prediksi yang identik.
3. Membandingkan CSV *submit* dengan prediksi TTA dan dengan prediksi tanpa TTA.
4. Menampilkan matriks silang (*crosstab*) dan distribusi kelas kedua hasil.

In [12]:
if not {"id", "predicted"}.issubset(submitted_raw.columns):
    print("Nama kolom tidak standar; dua kolom pertama dipakai sebagai id dan predicted.")
    submitted_raw = submitted_raw.iloc[:, :2].copy()
    submitted_raw.columns = ["id", "predicted"]

submitted = submitted_raw[["id", "predicted"]].astype(int)
reproduced = result_df[["id", "predicted"]].astype(int)

ids_submit, ids_repro = set(submitted["id"]), set(reproduced["id"])
print("Jumlah id CSV submit     :", len(ids_submit))
print("Jumlah id reproduksi     :", len(ids_repro))
print("Id hanya di CSV submit   :", len(ids_submit - ids_repro))
print("Id hanya di reproduksi   :", len(ids_repro - ids_submit))
if submitted["id"].duplicated().any():
    print("PERINGATAN: CSV submit memiliki id ganda.")

both = (
    submitted.rename(columns={"predicted": "submit"})
    .merge(reproduced.rename(columns={"predicted": "reproduksi"}), on="id", how="inner")
    .merge(result_df[["id", "pred_no_tta"]], on="id", how="left")
)

n_compared = len(both)
n_identical = int((both["submit"] == both["reproduksi"]).sum())
n_different = n_compared - n_identical
agreement = n_identical / n_compared if n_compared else float("nan")
agreement_no_tta = float((both["submit"] == both["pred_no_tta"]).mean()) if n_compared else float("nan")

print("\n=== Hasil perbandingan ===")
print(f"Jumlah citra dibandingkan            : {n_compared}")
print(f"Prediksi identik (TTA vs submit)     : {n_identical} ({fmt_pct(agreement)}%)")
print(f"Prediksi berbeda (TTA vs submit)     : {n_different}")
print(f"Kesesuaian tanpa TTA vs submit       : {fmt_pct(agreement_no_tta)}%")

print("\nMatriks silang (baris: CSV submit, kolom: reproduksi):")
crosstab = pd.crosstab(
    both["submit"].map(CLASS_NAMES), both["reproduksi"].map(CLASS_NAMES),
    rownames=["CSV submit"], colnames=["Reproduksi"],
)
display(crosstab)

dist = pd.DataFrame({
    "CSV submit": both["submit"].map(CLASS_NAMES).value_counts(),
    "Reproduksi": both["reproduksi"].map(CLASS_NAMES).value_counts(),
}).fillna(0).astype(int)
print("Distribusi kelas:")
display(dist)

Jumlah id CSV submit     : 1458
Jumlah id reproduksi     : 1458
Id hanya di CSV submit   : 0
Id hanya di reproduksi   : 0

=== Hasil perbandingan ===
Jumlah citra dibandingkan            : 1458
Prediksi identik (TTA vs submit)     : 1458 (100,0000%)
Prediksi berbeda (TTA vs submit)     : 0
Kesesuaian tanpa TTA vs submit       : 99,8628%

Matriks silang (baris: CSV submit, kolom: reproduksi):


Reproduksi,Electronic,Organic,Recyclable
CSV submit,,,
Electronic,201,0,0
Organic,0,703,0
Recyclable,0,0,554


Distribusi kelas:


,CSV submit,Reproduksi
Organic,703,703
Recyclable,554,554
Electronic,201,201


## 10. Analisis Perbedaan

Jika ada prediksi yang berbeda, tabel berikut menampilkan citra tersebut beserta probabilitas dan *margin*-nya.

Perbedaan yang sangat kecil dapat muncul karena perbedaan jenis GPU, versi pustaka, atau presisi campuran (FP16), terutama pada citra yang *margin*-nya rendah. Sebaliknya, perbedaan yang banyak, atau yang terjadi pada citra dengan *margin* tinggi, menunjukkan bahwa CSV *submit* dan *checkpoint* ini tidak berasal dari model yang sama.

In [13]:
diff = both[both["submit"] != both["reproduksi"]].merge(
    result_df[["id", "prob_recyclable", "prob_electronic", "prob_organic", "margin"]], on="id", how="left"
)
diff["submit_kelas"] = diff["submit"].map(CLASS_NAMES)
diff["reproduksi_kelas"] = diff["reproduksi"].map(CLASS_NAMES)

if diff.empty:
    print("Tidak ada perbedaan: seluruh prediksi identik dengan CSV submit.")
else:
    print(f"{len(diff)} citra memiliki prediksi berbeda.")
    print(f"Margin median citra berbeda        : {diff['margin'].median():.4f}")
    print(f"Margin median seluruh citra uji    : {result_df['margin'].median():.4f}")
    print(f"Citra berbeda dengan margin < 0,10 : {int((diff['margin'] < 0.10).sum())}")
    display(
        diff[["id", "submit_kelas", "reproduksi_kelas",
              "prob_recyclable", "prob_electronic", "prob_organic", "margin"]]
        .sort_values("margin")
        .reset_index(drop=True)
    )
    diff.to_csv(OUTPUT_DIR / "perbedaan_prediksi.csv", index=False)

Tidak ada perbedaan: seluruh prediksi identik dengan CSV submit.


## 11. Kesimpulan dan Laporan Verifikasi

Sel berikut mencetak kesimpulan berdasarkan angka hasil perbandingan, lalu menyimpan laporan `laporan_verifikasi.json` yang berisi SHA-256 *checkpoint*, metadata pelatihan, lingkungan eksekusi, dan hasil perbandingan. Seluruh berkas hasil dikemas dalam `hasil_verifikasi.zip` di `/kaggle/working`, dan dapat diunduh dari tab *Output* setelah notebook dijalankan dengan *Save Version*.

Batasan: kesesuaian hasil menunjukkan bahwa CSV *submit* dapat dibangkitkan dari *checkpoint* yang dilaporkan hanya dengan citra data uji. Notebook ini tidak menggantikan pemeriksaan kode pelatihan.

In [14]:
print("=== KESIMPULAN ===")
print(f"Checkpoint : {HF_REPO_ID}/{HF_CKPT_FILENAME}")
print(f"SHA-256    : {ckpt_sha256}")
print(f"Data uji   : {len(test_df)} citra; label data uji tidak dibaca.")
if n_compared == 0:
    print("Tidak ada id yang cocok antara CSV submit dan hasil reproduksi. Periksa kembali kedua berkas.")
elif n_different == 0:
    print(f"Seluruh {n_compared} prediksi identik dengan CSV submit (kesesuaian 100%).")
else:
    print(f"{n_identical} dari {n_compared} prediksi identik dengan CSV submit (kesesuaian {fmt_pct(agreement)}%).")
    print(f"{n_different} prediksi berbeda. Rincian pada bagian 10 dan berkas perbedaan_prediksi.csv.")

report = {
    "checkpoint": {
        "repo": HF_REPO_ID,
        "file": HF_CKPT_FILENAME,
        "sha256": ckpt_sha256,
        "size_bytes": ckpt_size_bytes,
        "epoch": train_config.get("epoch") if isinstance(train_config, dict) else None,
        "training_config": train_config,
    },
    "inference": {
        "n_test_images": int(len(test_df)),
        "tta_views": TTA_N_VIEWS,
        "batch_size": BATCH_SIZE,
        "device": str(DEVICE),
        "gpu": GPU_NAME,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "n_changed_by_tta": n_changed_by_tta,
    },
    "comparison": {
        "n_compared": int(n_compared),
        "n_identical": int(n_identical),
        "n_different": int(n_different),
        "agreement_pct": round(agreement * 100, 6) if n_compared else None,
        "agreement_no_tta_pct": round(agreement_no_tta * 100, 6) if n_compared else None,
        "ids_only_in_submit": len(ids_submit - ids_repro),
        "ids_only_in_reproduction": len(ids_repro - ids_submit),
    },
}
report_path = OUTPUT_DIR / "laporan_verifikasi.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
shutil.copy(SUBMITTED_CSV_PATH, OUTPUT_DIR / "submission_asli.csv")

zip_path = shutil.make_archive("/kaggle/working/hasil_verifikasi", "zip", OUTPUT_DIR)
print("\nLaporan  :", report_path)
print("Arsip    :", zip_path)

=== KESIMPULAN ===
Checkpoint : ayaayaa/train-val-best-test/train_val_best_test.pt
SHA-256    : 03aa480d1568ee5b5303428b146c13970e810f308bb6e39c5956949492acfe14
Data uji   : 1458 citra; label data uji tidak dibaca.
Seluruh 1458 prediksi identik dengan CSV submit (kesesuaian 100%).

Laporan  : /kaggle/working/hasil_verifikasi/laporan_verifikasi.json
Arsip    : /kaggle/working/hasil_verifikasi.zip
